Be sure to run this in Segger-incremental ENV

WATCH out to define output dir properly, depending on Use of Single Cell data


In [ ]:
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
single_cell_used = "no"  # "no" or "with"  --- IMPORTANT ---
INPUT_DIR = Path("")  # folder containing per-sample subdirs with boundaries and transcripts
SEGGER_OUTPUT_BASE_DIR = Path("")  # base output dir; subdir {single_cell_used}_SC_reference created automatically
SCRNA_REFERENCE_PATH = Path("")  # path to scRNA .h5ad reference (only needed if single_cell_used == "with")

# derived path (do not edit)
base_output_dir = SEGGER_OUTPUT_BASE_DIR / f"{single_cell_used}_SC_reference"


In [ ]:
import os
import subprocess

# Ensure the base output directory exists
base_output_dir.mkdir(parents=True, exist_ok=True)

my_env = os.environ.copy()
my_env["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 0. Change as needed.

# --- THE LOOP ---
for folder_name in os.listdir(INPUT_DIR):
    input_folder_path = INPUT_DIR / folder_name
    print(input_folder_path)

    if input_folder_path.is_dir():
        sample_id = folder_name

        sample_output_dir = base_output_dir / sample_id
        sample_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"🚀 Starting Segger for: {sample_id}")

        #"--node-representation-dim", "95",
        if single_cell_used == "with":
            command = [
                "segger", "segment",
                "--min-qv", "0",
                "--node-representation-dim", "60",
                "--prediction-mode", "nucleus",
                "--scrna-reference-path", str(SCRNA_REFERENCE_PATH),
                "-i", str(input_folder_path),
                "-o", str(sample_output_dir)
            ]
        else:
            command = [
                "segger", "segment",
                "--min-qv", "0",
                "--node-representation-dim", "60",
                "--prediction-mode", "nucleus",
                "-i", str(input_folder_path),
                "-o", str(sample_output_dir)
            ]

        try:
            print(f"Running Segger on {sample_id}...")
            result = subprocess.run(
                command,
                env=my_env,
                check=True,
                capture_output=True,
                text=True
            )
            print("Success!")
        except subprocess.CalledProcessError as e:
            print(f"Error: {e.stderr}")


And now we need to get Anndata object, there is Seger export option, but it is not exactly what we need. Instead
we first join the initial transcritps table with the predicted one, bcs the predicted one has only cell_id no location data.
Joining is done according to row indees, which remanin preserved 

In [9]:
def segger2adata(segger_output_folder, initial_transcripts_folder):
    # This function would read the Segger output and convert it to an AnnData object.
    #predictions are the output of segger, input_tx is the original transcripts.parquet file used as inpput for segger 
    from shapely.geometry import MultiPoint
    import anndata as ad
    import pandas as pd

    predictions = pd.read_parquet(segger_output_folder / "segger_segmentation.parquet")
    input_tx = pd.read_parquet(initial_transcripts_folder / "transcripts.parquet")

    merged = pd.merge(predictions, input_tx, left_on="row_index", right_index=True, how="left")

    #conversion to cell by gene matrix, for anndata object
    merged = merged[merged['segger_cell_id'].notna()] 
    #removing all transcripts that were assigned to cell that has less than 2 trasncripts assigned to it
    cell_counts = merged['segger_cell_id'].value_counts()
    valid_cells = cell_counts[cell_counts >= 2].index
    merged = merged[merged['segger_cell_id'].isin(valid_cells)]

    #adding N columns, wher N is the number of genes sampled, 
    #value for each cell will be SUM of all transcripts of the same gene in them
    cell_gene_matrix = pd.crosstab(merged['segger_cell_id'], merged['gene'])

    #calculating NEW cetnroids for each cell
    centroids = (
        merged
        .groupby('segger_cell_id')
        .apply(lambda g: MultiPoint(g[['global_x', 'global_y']].to_numpy()).convex_hull.centroid, include_groups=False)
    )


    # Convert to DataFrame
    centroids_df = centroids.apply(lambda p: (p.x, p.y)).to_list()
    centroids_df = pd.DataFrame(centroids_df, index=centroids.index, columns=['centroid_x', 'centroid_y'])

    cell_gene_matrix = centroids_df.join(cell_gene_matrix, on='segger_cell_id') #joining matrix with centorids

    #generating anndata object

    # cell metadata
    obs = cell_gene_matrix[['centroid_x', 'centroid_y']]

    # gene expression matrix
    X = cell_gene_matrix.drop(columns=['centroid_x', 'centroid_y'])
    X = X.fillna(0)
    # gene metadata
    var = pd.DataFrame(index=X.columns)

    # create AnnData object
    adata = ad.AnnData(X=X, obs=obs, var=var)

    #final touches needed for Segtraq compatibiltiy
    adata.obs["cell_id"] = adata.obs.index.astype(int)  # segger_cell_id is actually INDEX here, and segtraq exepcts a column called cell_id not segger_cell_id
    adata.obs["region"] = "cell_boundaries"
    adata.obs["region"] = adata.obs["region"].astype("category")

    # add spatial coordinates to obsm
    adata.obsm['spatial'] = obs[['centroid_x', 'centroid_y']].values

    merged['cell_id'] = merged['segger_cell_id']

    merged.to_parquet(segger_output_folder / "merged_segger_segmentation.parquet") #saving the merged dataframe for reference
    adata.write_h5ad(segger_output_folder / "segger_adata.h5ad")
    print(f"finished converting {segger_output_folder} to anndata object")


Main script for conversion of segger parquet output file into Anndata object

In [ ]:
import os
from pathlib import Path

segger_output_folder = SEGGER_OUTPUT_BASE_DIR / f"{single_cell_used}_SC_reference"
segger_input_folder = INPUT_DIR

for folder_name in os.listdir(segger_output_folder):
    sample_folder_path = segger_output_folder / folder_name
    #print(sample_folder_path)
    #print(folder_name)

    segger2adata(sample_folder_path, segger_input_folder / folder_name)


Furher postprocessing for Segtraq compatibility

In [ ]:
def cell_to_polygon(g):
    import geopandas as gpd
    from shapely.geometry import MultiPoint
    return MultiPoint(g[['global_x', 'global_y']].to_numpy()).convex_hull  #same method as in 2cellbygene.ipynb

In [19]:
def to_spatialdata(sample, input_folder):
    import spatialdata as sd
    import anndata as ad
    import pandas as pd
    import geopandas as gpd

    #loading the anndata object and the transcripts dataframe
    adata = ad.read_h5ad(f'{input_folder}/{sample}/segger_adata.h5ad')
    transcripts_df = pd.read_parquet(f'{input_folder}/{sample}/merged_segger_segmentation.parquet')

    #boundary file generation
    cell_shapes = (
        transcripts_df
        .groupby("cell_id")
        .apply(cell_to_polygon)
    )
    cell_shapes_gdf = gpd.GeoDataFrame(
        geometry=cell_shapes,
        index=cell_shapes.index
    )
    cell_shapes_gdf.index.name = "cell_id"

    #transcripts file adjustements
    transcripts_df = transcripts_df.rename(columns={
        "gene": "feature_name",
        "global_x": "x",
        "global_y": "y",
        "global_z": "z"
        
    })

    transcripts_df["feature_name"] = transcripts_df["feature_name"].astype("category")
    
    #adata adjustements

    #print(f'Number of cells in adata: {len(adata.obs)}')
    #print(f'Number of cells in cell_shapes_gdf: {len(cell_shapes_gdf)}')  
    adata.obs["cell_id"] = cell_shapes_gdf.index.astype(str) #segtraq needs adata and cell_shapes_gdf to have the same index, which is cell_id, so we need to make sure they match. This line is just a check, it should return True for all cells.
    cell_shapes_gdf.index = cell_shapes_gdf.index.astype(str)
    adata.obs_names = adata.obs["cell_id"]

    print(cell_shapes_gdf.geometry.geom_type.value_counts())
    
    #creating the spatialdata object
    sdata = sd.SpatialData(
        points={
            "transcripts": sd.models.PointsModel.parse(transcripts_df)
        },
        shapes={
            "cell_boundaries": sd.models.ShapesModel.parse(cell_shapes_gdf)
        },
        tables={
            "table": sd.models.TableModel.parse(
                adata,
                region_key="region",
                region="cell_boundaries",
                instance_key="cell_id"
            )
        },
    )

    sdata.write(f"{input_folder}/{sample}/segger_spatialdata.zarr", overwrite=True)

In [20]:
single_cell_used = 'no' # 'no' / 'with'

In [ ]:
# --CONFIG--
segger_output_folder = SEGGER_OUTPUT_BASE_DIR / f"{single_cell_used}_SC_reference"

for sample in os.listdir(segger_output_folder):
    print(f"Processing sample: {sample}")
    to_spatialdata(sample, str(segger_output_folder))


This is the end of postprocessing SEgger ouputs, next step is integrating all data in Segtraq anaylsis